In [1]:
import requests
from pathlib import Path

class APIClient:
    def __init__(self, base_url, username, password):
        self.base_url = base_url
        self.username = username
        self.password = password
        self.access_token = None
        self.refresh_token = None

    # ---------- AUTH ----------
    def authenticate(self):
        url = f"{self.base_url}/auth/token"
        payload = {
            "username": self.username,
            "password": self.password
        }

        r = requests.post(url, json=payload, timeout=10)
        r.raise_for_status()

        data = r.json()
        self.access_token = data["access_token"]
        self.refresh_token = data["refresh_token"]

    def refresh(self):
        url = f"{self.base_url}/auth/refresh"
        payload = {
            "refresh_token": self.refresh_token
        }

        r = requests.post(url, json=payload, timeout=10)
        r.raise_for_status()

        data = r.json()
        self.access_token = data["access_token"]

    # ---------- UTILS ----------
    def headers(self):
        return {
            "Authorization": f"Bearer {self.access_token}"
        }

    def request_with_refresh(self, method, url, **kwargs):
        r = requests.request(method, url, headers=self.headers(), **kwargs)

        if r.status_code == 401:
            print("🔁 Token expiré → refresh")
            self.refresh()
            r = requests.request(method, url, headers=self.headers(), **kwargs)

        return r

    # ---------- UPLOAD ----------
    def upload_image(self, image_path):
        # CF-495-020-000010-PPDef.pdf → 495-020-000010
        request = image_path.stem.replace("CF-", "").replace("-PPDef", "")
        #/api/v1/bo/certificates/{orderNumber}/documents
        
        url = f"{self.base_url}/bo/certificates/{request}/documents"

        with open(image_path, "rb") as img:
            files = {
                "file": (image_path.name, img, "application/pdf")
            }
            data = {
                "request": request 
            }

            r = self.request_with_refresh(
                "POST",
                url,
                files=files,
                data=data,
                timeout=30
            )

        r.raise_for_status()
        return r.json()


In [2]:
client = APIClient(
    base_url="https://collect.prod.digifor.ci/api/v1",
    username="sifor.tonkpi@gmail.com",
    password="Wj!7rPz#2vB9kAq"
)

client.authenticate()


In [3]:
IMAGE_DIR = Path(r"C:\Users\L14\Downloads\Plan du 22-03-26")

images = sorted(
    list(IMAGE_DIR.glob("*.jpg")) +
    list(IMAGE_DIR.glob("*.png")) +
    list(IMAGE_DIR.glob("*.pdf"))
)

In [4]:
import csv
from datetime import datetime

success_rows = []
failed_rows = []

for img_path in images:
    request_number = img_path.stem
    now = datetime.now().isoformat()

    try:
        result = client.upload_image(img_path)

        success_rows.append({
            "file": img_path.name,
            "request": request_number,
            "timestamp": now,
        })

        print(f"✅ {img_path.name} envoyé")

    except Exception as e:
        failed_rows.append({
            "file": img_path.name,
            "request": request_number,
            "error": str(e),
            "timestamp": now,
        })

        print(f"❌ {img_path.name} → {e}")

# Write success.csv
with open("success.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file", "request", "timestamp"])
    writer.writeheader()
    writer.writerows(success_rows)

# Write failed.csv
with open("failed.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file", "request", "error", "timestamp"])
    writer.writeheader()
    writer.writerows(failed_rows)

✅ CF-463-003-000006-PPDef.pdf envoyé
✅ CF-463-003-000007-PPDef.pdf envoyé
✅ CF-463-003-000008-PPDef.pdf envoyé
✅ CF-463-003-000013-PPDef.pdf envoyé
✅ CF-463-003-000014-PPDef.pdf envoyé
✅ CF-463-003-000016-PPDef.pdf envoyé
✅ CF-463-003-000018-PPDef.pdf envoyé
✅ CF-463-003-000019-PPDef.pdf envoyé
✅ CF-463-003-000020-PPDef.pdf envoyé
✅ CF-463-003-000023-PPDef.pdf envoyé
✅ CF-463-003-000028-PPDef.pdf envoyé
✅ CF-463-003-000029-PPDef.pdf envoyé
✅ CF-463-003-000030-PPDef.pdf envoyé
✅ CF-463-003-000035-PPDef.pdf envoyé
✅ CF-463-005-000001-PPDef.pdf envoyé
✅ CF-463-005-000005-PPDef.pdf envoyé
✅ CF-463-005-000011-PPDef.pdf envoyé
✅ CF-463-005-000012-PPDef.pdf envoyé
✅ CF-463-005-000015-PPDef.pdf envoyé
✅ CF-463-005-000017-PPDef.pdf envoyé
✅ CF-463-005-000018-PPDef.pdf envoyé
✅ CF-463-005-000020-PPDef.pdf envoyé
✅ CF-463-005-000021-PPDef.pdf envoyé
✅ CF-463-005-000023-PPDef.pdf envoyé
✅ CF-463-005-000025-PPDef.pdf envoyé
✅ CF-463-005-000028-PPDef.pdf envoyé
✅ CF-463-005-000029-PPDef.pdf envoyé
✅